# Milestone 3 - Data Preprocessing

**Goal:** turn the clean Milestone 2 data into a dataset that is ready for modeling: every column numeric, scaled sensibly, and split into train / test with **no overlap and no leakage**.

**The question we will model (decided from the EDA):**
> At the moment a flight leaves the gate late, will it still arrive **15+ minutes late**?

**Why this question:** our data only contains flights that departed 6+ minutes late, so this is the question it can honestly answer. It is also a real decision point for a dispatch team: they know the departure delay at pushback and can still protect connections and the aircraft's next flights.

Order of the pipeline (and why this order):
1. Keep completed flights and create the target
2. Feature engineering (new columns, each with a reason)
3. Choose columns: drop leakage, IDs and redundant columns
4. Take a random sample
5. **Split into train / test** (before any encoding or scaling, see the reason in step 5)
6. Group rare airports (learned from train only)
7. One-hot encode category columns (fit on train only)
8. Scale numeric columns (fit on train only)
9. Final readiness checks
10. Save the final files

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('flights_2008_clean.csv')
print('clean data from Milestone 2:', df.shape)

clean data from Milestone 2: (1936743, 33)


## Step 1 - Keep completed flights and create the target

**Why only completed flights:** cancelled and diverted flights have no arrival delay, so they have no target value. We decided in Milestone 2 not to invent one.

**Target `IsDelayed`:** 1 if the flight arrived 15+ minutes late, otherwise 0.
**Why 15 minutes:** it is the official US DOT definition of a delayed flight.

We also keep `ArrDelay` in minutes. It is **not** an input. We save it next to the target because in Milestone 4 we also try to predict *how many minutes* late a flight will be.

In [2]:
flights = df[(df['Cancelled'] == 0) & (df['Diverted'] == 0)].copy()
flights['IsDelayed'] = (flights['ArrDelay'] >= 15).astype(int)

# keep the original row number as an ID so we can check later that train and test don't overlap
flights['FlightID'] = flights.index

print('completed flights:', len(flights))
print('delayed share:', round(flights['IsDelayed'].mean() * 100, 1), '%')

completed flights: 1928361
delayed share: 64.7 %


## Step 2 - Feature engineering

Each new feature has a reason that comes from the EDA notebook.

| New feature | How we make it | Why |
|---|---|---|
| `DepHour` | hour of `CRSDepTime` (1755 -> 17) | hhmm is not a real number (12:59 is stored as 1259 and 13:00 as 1300, a jump of 41 for one minute). The EDA showed a clear rise in delays from 6 am (~58%) to 6-9 pm (~69%), so the hour is the useful part |
| `Season` | Winter (Dec-Feb), Spring (Mar-May), Summer (Jun-Aug), Fall (Sep-Nov) | The month pattern goes up and down (Dec ~70%, Oct ~55%, June ~68%), so the month number as a straight scale (1 to 12) would mislead linear models. Seasons group months that behave alike |
| `LogDepDelay` | log of `DepDelay` | `DepDelay` is very right-skewed (from 6 to over 2,400 min). After about 60 minutes the arrival is late almost 100% of the time anyway, so the difference between 300 and 600 minutes doesn't matter. The log squeezes the long tail so a few huge values don't dominate scaling and distance-based models like KNN |

**Why `DepDelay` is allowed as an input:** we predict at the moment the plane leaves the gate, and the departure delay is known at that moment. It is not leakage for our question. (In Milestone 4 we also test a model *without* it, to show how much it matters.)

**Leak check for this step:** each new feature uses only values from its own row, so nothing is learned from other flights and nothing can leak between train and test.

In [3]:
flights['DepHour'] = (flights['CRSDepTime'] // 100) % 24

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

flights['Season'] = flights['Month'].apply(get_season)

# DepDelay is always 6 or more in this dataset, so the log is always defined
flights['LogDepDelay'] = np.log(flights['DepDelay'])

print('DepDelay skew   :', round(flights['DepDelay'].skew(), 2))
print('LogDepDelay skew:', round(flights['LogDepDelay'].skew(), 2))
print()
print('% delayed by season:')
print((flights.groupby('Season')['IsDelayed'].mean() * 100).round(1))

DepDelay skew   : 4.5
LogDepDelay skew: 0.38

% delayed by season:


Season
Fall      58.2
Spring    63.7
Summer    65.5
Winter    68.2
Name: IsDelayed, dtype: float64


**Result:** the log transform brings the skew from about 4.5 down to about 0.4 (0 would be perfectly symmetric). The seasons show the pattern we wanted to capture: Winter is the worst, Fall the best.

## Step 3 - Choose the columns

**Inputs we keep (8 columns):**

| Column | Type | Why we keep it |
|---|---|---|
| `LogDepDelay` | numeric | strongest signal in the EDA (22% delayed at 6-14 min vs 96%+ at 30+ min) |
| `DepHour` | numeric | delays build up during the day |
| `Distance` | numeric | short flights are delayed more often (68% vs 59%) |
| `Season` | category | seasonal pattern |
| `DayOfWeek` | category | small effect (Fri 66.5% vs Sat 61.5%); we let the model decide |
| `UniqueCarrier` | category | big differences between airlines (54% to 76%) |
| `Origin` | category | busy, cold-weather hubs are worse |
| `Dest` | category | the arrival airport's congestion also affects arrival delay |

**Columns we drop, and why:**

| Dropped column(s) | Reason |
|---|---|
| `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` | **Leakage**: they add up exactly to `ArrDelay` and are only known after landing |
| `ArrTime`, `ActualElapsedTime`, `AirTime`, `TaxiIn`, `TaxiOut` | **Leakage**: only known after departure or after landing |
| `ExtremeDelay` | **Leakage**: built from `ArrDelay` (the answer) |
| `DepTime` | same information as `DepDelay` + `CRSDepTime` |
| `Cancelled`, `Diverted`, `CancellationCode` | always 0 / "N" after step 1, so they carry no information |
| `CRSElapsedTime` | 0.98 correlation with `Distance`, almost a copy (and `CRSElapsedTimeFilled` goes with it) |
| `CRSArrTime` | about the same as departure hour + flight length, which we already have |
| `CRSDepTime`, `Month`, `DepDelay` | replaced by `DepHour`, `Season`, `LogDepDelay` |
| `FlightNum`, `TailNum` | **ID columns**: a flight number is a label, not a quantity, and thousands of IDs would make the model memorize instead of learn |
| `Year` | always 2008 |
| `DayofMonth`, `FlightDate` | with month + day the model could memorize **specific bad days** (the delayed share by date ranges from 43% to 83%). In a random split, the same bad day is in train and test, so the score would look better than it really is for future flights, whose weather we can't know in advance |
| `SuspectAirTime` | quality flag for `AirTime`, which we don't use |

In [4]:
input_cols = ['LogDepDelay', 'DepHour', 'Distance', 'Season', 'DayOfWeek', 'UniqueCarrier', 'Origin', 'Dest']

# we also carry the ID, both targets and the raw DepDelay along (they are NOT inputs)
data = flights[['FlightID'] + input_cols + ['DepDelay', 'IsDelayed', 'ArrDelay']].copy()

print('input columns:', len(input_cols))
print('missing values in the kept columns:', int(data.isna().sum().sum()))
data.head()

input columns: 8


missing values in the kept columns: 0


,FlightID,LogDepDelay,DepHour,Distance,Season,DayOfWeek,UniqueCarrier,Origin,Dest,DepDelay,IsDelayed,ArrDelay
0,0,2.079442,19,810,Winter,4,WN,IAD,TPA,8,0,-14.0
1,1,2.944439,7,810,Winter,4,WN,IAD,TPA,19,0,2.0
2,2,2.079442,6,515,Winter,4,WN,IND,BWI,8,0,14.0
3,3,3.526361,17,515,Winter,4,WN,IND,BWI,34,1,34.0
4,4,3.218876,19,688,Winter,4,WN,IND,JAX,25,0,11.0


## Step 4 - Take a random sample of 100,000 flights

**Why:** we have about 1.93 million completed flights. That's more than we need to learn these patterns, and some Milestone 4 models (especially KNN, which compares each flight with every training flight) would take hours on a normal laptop. With 100,000 random flights every model can be trained and cross-validated in a few minutes, and the results stay stable.

**Why random (with a fixed seed):** a random sample keeps the same mix of airlines, months and delays as the full data. `random_state=42` makes the result the same every time anyone on the team runs the notebook.

In [5]:
data = data.sample(n=100000, random_state=42)

print('sample size:', len(data))
print('delayed share in sample   :', round(data['IsDelayed'].mean() * 100, 1), '%')
print('delayed share in full data:', round(flights['IsDelayed'].mean() * 100, 1), '%')

sample size: 100000
delayed share in sample   : 64.8 %
delayed share in full data: 64.7 %


## Step 5 - Split into train and test (80 / 20)

**Why split now, before encoding and scaling:** the test set has to act like *new flights we have never seen*. If we fit the scaler or choose the airport groups using all the data, information from the test set (its averages, its airports) leaks into training, and the test score becomes too optimistic. So from here on, everything is **learned from the train set only** and then applied to the test set.

**Why 80 / 20:** it is the common choice. 80,000 flights are enough to train on, and 20,000 test flights give a stable score.

**Why `stratify`:** it keeps the same delayed share (about 65%) in both sets, so train and test are fair copies of each other.

In [6]:
X = data[['FlightID'] + input_cols]
y = data[['IsDelayed', 'ArrDelay']]
details = data[['FlightID', 'DepDelay', 'UniqueCarrier', 'Origin', 'DepHour']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y['IsDelayed'])

print('train rows:', len(X_train), '| test rows:', len(X_test))
print('delayed share train:', round(y_train['IsDelayed'].mean() * 100, 1), '% | test:', round(y_test['IsDelayed'].mean() * 100, 1), '%')

# no flight is allowed to be in both sets
overlap = set(X_train['FlightID']) & set(X_test['FlightID'])
print('flights in both train and test:', len(overlap))

train rows: 80000 | test rows: 20000
delayed share train: 64.8 % | test: 64.8 %
flights in both train and test: 0


## Step 6 - Group rare airports into "Other"

**Why:** there are about 300 airports. Many have only a handful of flights, which is too few to learn anything reliable about them, and one column per airport would add hundreds of mostly-empty columns. We keep the **15 busiest origin and 15 busiest destination airports** (together they cover about half of all flights) and call the rest "Other".

**Leak check:** the list of busiest airports is counted on the **train set only**, then the same list is used for the test set.

In [7]:
X_train = X_train.copy()
X_test = X_test.copy()

for col in ['Origin', 'Dest']:
    top15 = X_train[col].value_counts().head(15).index
    print(col, 'kept:', list(top15))
    X_train[col] = X_train[col].where(X_train[col].isin(top15), 'Other')
    X_test[col] = X_test[col].where(X_test[col].isin(top15), 'Other')

print()
print('train flights now labeled "Other" origin:', round((X_train['Origin'] == 'Other').mean() * 100, 1), '%')

Origin kept: ['ATL', 'ORD', 'DFW', 'DEN', 'LAX', 'IAH', 'PHX', 'EWR', 'LAS', 'DTW', 'SFO', 'JFK', 'MCO', 'CLT', 'MSP']
Dest kept: ['ORD', 'ATL', 'DFW', 'DEN', 'LAX', 'EWR', 'LAS', 'PHX', 'SFO', 'IAH', 'LGA', 'MCO', 'JFK', 'DTW', 'SEA']

train flights now labeled "Other" origin: 51.8 %


## Step 7 - One-hot encode the category columns

**Why one-hot and not label encoding:** label encoding would turn airlines into numbers like AA=0, WN=18, YV=19, and a model would think YV is "more" than AA, or that WN is between them. Airlines, airports, seasons and weekdays have **no natural order**, so one-hot encoding (one 0/1 column per value) is the correct choice.

`DayOfWeek` is stored as 1-7, but it is also a category (Sunday is not "7 times Monday"), so it is one-hot encoded too.

**Leak check:** the encoder is fit on train only. `handle_unknown='ignore'` means a value that appears only in the test set would simply get all zeros instead of an error.

In [8]:
cat_cols = ['Season', 'DayOfWeek', 'UniqueCarrier', 'Origin', 'Dest']
num_cols = ['LogDepDelay', 'DepHour', 'Distance']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[cat_cols].astype(str))

new_names = encoder.get_feature_names_out(cat_cols)
train_cat = pd.DataFrame(encoder.transform(X_train[cat_cols].astype(str)), columns=new_names, index=X_train.index)
test_cat = pd.DataFrame(encoder.transform(X_test[cat_cols].astype(str)), columns=new_names, index=X_test.index)

print(len(cat_cols), 'category columns became', len(new_names), 'numeric 0/1 columns')
print(list(new_names[:12]), '...')

5 category columns became 63 numeric 0/1 columns
['Season_Fall', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'DayOfWeek_1', 'DayOfWeek_2', 'DayOfWeek_3', 'DayOfWeek_4', 'DayOfWeek_5', 'DayOfWeek_6', 'DayOfWeek_7', 'UniqueCarrier_9E'] ...


## Step 8 - Scale the numeric columns

**Why:** the numeric columns have very different ranges (`Distance` goes up to ~5,000 miles, `DepHour` 0-23, `LogDepDelay` about 1.8-7.8). Models that use distances (KNN) or penalties (Logistic Regression) would treat the column with the biggest numbers as the most important one. `StandardScaler` gives each numeric column mean 0 and standard deviation 1, so they are compared fairly.

**Why we don't scale the one-hot columns:** they are already 0/1, which is a sensible scale, and keeping them as 0/1 keeps them easy to read ("this flight is an AA flight").

**Leak check:** the scaler learns the mean and std from the **train set only**. The test set is transformed with the train numbers, which is why its mean is close to 0 but not exactly 0.

In [9]:
scaler = StandardScaler()
scaler.fit(X_train[num_cols])

train_num = pd.DataFrame(scaler.transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
test_num = pd.DataFrame(scaler.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

print('train means:', train_num.mean().round(3).to_dict())
print('train stds :', train_num.std().round(3).to_dict())
print('test means :', test_num.mean().round(3).to_dict())

X_train_final = pd.concat([train_num, train_cat], axis=1)
X_test_final = pd.concat([test_num, test_cat], axis=1)

train means: {'LogDepDelay': -0.0, 'DepHour': -0.0, 'Distance': -0.0}
train stds : {'LogDepDelay': 1.0, 'DepHour': 1.0, 'Distance': 1.0}
test means : {'LogDepDelay': 0.007, 'DepHour': 0.002, 'Distance': 0.004}


## Step 9 - Final readiness checks

**Why:** the most common Milestone 3 mistake is to assume the data is ready without checking. We check every rule of "ready for modeling" in code:
1. every column is numeric
2. no missing values
3. train and test have the same columns in the same order
4. no flight is in both train and test, and no ID column is left in the inputs
5. no input is suspiciously close to the target (a leakage warning sign)

In [10]:
print('1. all columns numeric (train):', all(pd.api.types.is_numeric_dtype(t) for t in X_train_final.dtypes))
print('   all columns numeric (test) :', all(pd.api.types.is_numeric_dtype(t) for t in X_test_final.dtypes))
print('2. missing values train / test:', int(X_train_final.isna().sum().sum()), '/', int(X_test_final.isna().sum().sum()))
print('3. same columns in same order :', list(X_train_final.columns) == list(X_test_final.columns))
print('4. overlapping flights        :', len(set(X_train['FlightID']) & set(X_test['FlightID'])))
print('   ID columns left in inputs  :', [c for c in X_train_final.columns if c in ['FlightID', 'FlightNum', 'TailNum']])
print('shapes  train:', X_train_final.shape, ' test:', X_test_final.shape)

1. all columns numeric (train): True
   all columns numeric (test) : True
2. missing values train / test: 0 / 0
3. same columns in same order : True
4. overlapping flights        : 0
   ID columns left in inputs  : []
shapes  train: (80000, 66)  test: (20000, 66)


In [11]:
# 5. leakage warning sign: an input that is almost perfectly correlated with the target
target_corr = X_train_final.corrwith(y_train['IsDelayed']).abs().sort_values(ascending=False)
print('inputs most correlated with the target (train):')
print(target_corr.head(6).round(3))

inputs most correlated with the target (train):
LogDepDelay         0.646
UniqueCarrier_WN    0.109
DepHour             0.067
Season_Fall         0.064
Distance            0.058
UniqueCarrier_YV    0.052
dtype: float64


**Result:** the strongest input is `LogDepDelay` at about 0.65. That is strong, but far from 1.0, and we already know why it is strong (section 3 of the EDA) and why it is allowed (it is known at departure). The next strongest input (Southwest, `UniqueCarrier_WN`) is only about 0.11, so there is no sign of a hidden leak.

## Before-and-after snapshot

In [12]:
snapshot = pd.DataFrame({
    'Before': [f'{df.shape[1]} columns',
               '3 text columns (airline, origin, destination)',
               'raw units (minutes, miles, hhmm)',
               f'{len(df):,} rows, no split'],
    'After': [f'{X_train_final.shape[1]} input columns (8 chosen inputs, leakage/ID/redundant dropped)',
              f'{len(cat_cols)} category columns -> {len(new_names)} one-hot 0/1 columns',
              '3 numeric columns standardized (mean 0, std 1, fit on train)',
              f'train {len(X_train_final):,} / test {len(X_test_final):,} (no overlap)']},
    index=['Columns', 'Category columns', 'Feature scale', 'Rows'])
snapshot

,Before,After
Columns,33 columns,"66 input columns (8 chosen inputs, leakage/ID/..."
Category columns,"3 text columns (airline, origin, destination)",5 category columns -> 63 one-hot 0/1 columns
Feature scale,"raw units (minutes, miles, hhmm)","3 numeric columns standardized (mean 0, std 1,..."
Rows,"1,936,743 rows, no split","train 80,000 / test 20,000 (no overlap)"


## Step 10 - Save the final dataset

We save:
- `X_train.csv`, `X_test.csv`: the model inputs (all numeric)
- `y_train.csv`, `y_test.csv`: `IsDelayed` (main target) and `ArrDelay` in minutes (for the regression part of Milestone 4)
- `train_details.csv`, `test_details.csv`: readable values (real departure delay, airline, airport, hour) matched by row. They are **not** model inputs. We need them in Milestone 4 for the simple "rule" baseline and to explain where the model makes mistakes.

**Why files in a `data/` folder:** the modeling notebook then starts from exactly the same split every time, and nobody can accidentally re-split or re-scale with the test set. The folder is not uploaded to GitHub (see `.gitignore`) because it can be recreated by running this notebook.

In [13]:
os.makedirs('data', exist_ok=True)

X_train_final.to_csv('data/X_train.csv', index=False)
X_test_final.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)
details.loc[X_train.index].to_csv('data/train_details.csv', index=False)
details.loc[X_test.index].to_csv('data/test_details.csv', index=False)

print('saved files:', sorted(os.listdir('data')))

saved files: ['X_test.csv', 'X_train.csv', 'test_details.csv', 'train_details.csv', 'y_test.csv', 'y_train.csv']


## Summary

- **Question:** once a flight leaves late, will it arrive 15+ minutes late? (65% of flights in our data do)
- **Features:** 8 inputs: 3 engineered (`DepHour`, `Season`, `LogDepDelay`), each with a reason from the EDA
- **Leakage removed:** delay causes, taxi times, actual times, `ExtremeDelay`, and specific dates
- **Order:** sample -> split -> group airports -> encode -> scale, with everything learned from train only
- **Result:** fully numeric, no missing values, standardized numeric columns, 80,000 / 20,000 split with no overlap

Known limits we carry into Milestone 4:
- The data only has late-departing flights, so the model is not meant for flights that leave on time.
- The split is random. A time-based split (train on earlier months, test on later months) would be an even stricter test, and we list it as a planned improvement.